# 07 — Metadata & Provenance

A RAG system should not only know **what a chunk says**.

It should also know:

- where the chunk came from,
- which document produced it,
- which page or section produced it,
- which version of the source was processed,
- how the content was extracted,
- and enough information to reproduce or debug the ingestion result.

That information is **metadata and provenance**.

In this notebook we will use the documents in `data/` and build provenance from information that actually exists in those files and from events that happen during ingestion.

We will keep an important distinction throughout:

> **Document metadata** comes from the source document or file.

> **Ingestion metadata** comes from our pipeline.

> **Provenance** records the relationship between the resulting content and its source.

## Learning objectives

By the end of this notebook you should be able to:

1. Distinguish document metadata from ingestion metadata.
2. Extract metadata from PDF, DOCX, and HTML sources.
3. Preserve page-level provenance for PDF content.
4. Generate stable identifiers for source documents.
5. Track the extraction method and parser version.
6. Attach provenance to extracted content before chunking.
7. Understand why provenance should survive all the way to retrieval and generation.
8. Design metadata that supports debugging, citations, re-ingestion, and document versioning.
9. Avoid putting volatile or unnecessarily large metadata into vector payloads.

## 1. Install libraries

In [1]:
!pip install -q beautifulsoup4 python-docx pymupdf pymupdf4llm


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


## 2. The documents we will use

These are real assets already used in the ingestion notebooks:

```text
data/
├── employee_travel_policy.pdf
├── internal_rag_engineering_guide.md
├── q3_customer_support_operating_brief.docx
└── support_refunds.html
```

We will inspect their actual contents and file properties rather than inventing example document metadata.

In [2]:
from pathlib import Path
import hashlib
import json
import mimetypes
import platform
import sys
from datetime import datetime, timezone

DATA_DIR = Path("./data")

DOCUMENTS = [
    DATA_DIR / "employee_travel_policy.pdf",
    DATA_DIR / "internal_rag_engineering_guide.md",
    DATA_DIR / "q3_customer_support_operating_brief.docx",
    DATA_DIR / "support_refunds.html",
]

for path in DOCUMENTS:
    assert path.exists(), f"Missing document: {path}"

for path in DOCUMENTS:
    print(path, "→", path.stat().st_size, "bytes")

data/employee_travel_policy.pdf → 44745 bytes
data/internal_rag_engineering_guide.md → 1175 bytes
data/q3_customer_support_operating_brief.docx → 1005174 bytes
data/support_refunds.html → 1434 bytes


## 3. Why provenance is different from metadata

The terms are related, but they are not interchangeable.

### Metadata

Metadata describes an object.

For example:

```text
filename
file type
file size
document title
document version
owner
creation date
```

### Provenance

Provenance describes **where a particular piece of content came from** and how it was produced.

For example:

```text
chunk
  ↓
document_id = ...
  ↓
page = 3
  ↓
section = Refund Handling
  ↓
extraction_method = pymupdf4llm
  ↓
parser_version = ...
```

The provenance chain lets us move from a retrieved chunk back toward the original source.

## 4. File-level metadata

Start with metadata supplied by the filesystem.

This is ingestion metadata, not document content.

In [3]:
def file_metadata(path: Path) -> dict:
    stat = path.stat()

    return {
        "filename": path.name,
        "extension": path.suffix.lower(),
        "size_bytes": stat.st_size,
        "mime_type": mimetypes.guess_type(path.name)[0],
        "modified_at": datetime.fromtimestamp(
            stat.st_mtime, tz=timezone.utc
        ).isoformat(),
    }

for path in DOCUMENTS:
    print(json.dumps(file_metadata(path), indent=2))

{
  "filename": "employee_travel_policy.pdf",
  "extension": ".pdf",
  "size_bytes": 44745,
  "mime_type": "application/pdf",
  "modified_at": "2026-08-30T15:15:53.263316+00:00"
}
{
  "filename": "internal_rag_engineering_guide.md",
  "extension": ".md",
  "size_bytes": 1175,
  "mime_type": null,
  "modified_at": "2026-08-30T10:59:35.985233+00:00"
}
{
  "filename": "q3_customer_support_operating_brief.docx",
  "extension": ".docx",
  "size_bytes": 1005174,
  "mime_type": null,
  "modified_at": "2026-08-30T12:46:18.556242+00:00"
}
{
  "filename": "support_refunds.html",
  "extension": ".html",
  "size_bytes": 1434,
  "mime_type": "text/html",
  "modified_at": "2026-08-30T10:59:35.994233+00:00"
}


### Important distinction

Filesystem timestamps describe the **file in the ingestion environment**.

They do not necessarily describe when the document itself was authored or approved.

That distinction becomes important when documents are copied, exported, downloaded, or re-uploaded.

## 5. Stable document identifiers

A filename is not a reliable document identifier.

The same document can be renamed:

```text
policy.pdf
travel-policy.pdf
travel-policy-final.pdf
```

A content hash gives us a deterministic identifier for the exact bytes we processed.

In [4]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        while chunk := file.read(chunk_size):
            digest.update(chunk)

    return digest.hexdigest()

for path in DOCUMENTS:
    print(path.name)
    print("SHA-256:", sha256_file(path))

employee_travel_policy.pdf
SHA-256: e269b160c4cb9ae314a3efadf7285d33829c6d661abb81ea09b95d107b10c9a9
internal_rag_engineering_guide.md
SHA-256: f29522a1b6250c3de09070ef0858d259b6b24b981e3649910514f5e036385035
q3_customer_support_operating_brief.docx
SHA-256: 387b08c1667bb69ae80c067266757fa17a5d75beae7ac2079fd25f1f28e2045e
support_refunds.html
SHA-256: a496b95da60f57adc9f0cb9ed5faee85a014e8ad3028b971ee2a6aad8e0c6725


### What the hash tells us

A SHA-256 content hash answers:

> Did I process exactly these file bytes?

It does **not** answer:

> Is this document logically the same document?

Two files can contain the same logical document but differ in metadata, PDF generation details, compression, or other bytes.

Later, a production system may therefore use both:

- a **document identity** for the logical source, and
- a **content hash** for the exact artifact processed.

## 6. PDF document metadata

Now inspect metadata from the PDF itself.

In [5]:
import pymupdf

pdf_path = DATA_DIR / "employee_travel_policy.pdf"

with pymupdf.open(pdf_path) as doc:
    pdf_metadata = doc.metadata
    page_count = len(doc)

print("Page count:", page_count)
print(json.dumps(pdf_metadata, indent=2))

Page count: 1
{
  "format": "PDF 1.4",
  "title": "",
  "author": "",
  "subject": "",
  "keywords": "",
  "creator": "",
  "producer": "",
  "creationDate": "",
  "modDate": "",
  "trapped": "",
  "encryption": null
}


## 7. DOCX document metadata and content metadata

DOCX files can contain package/core properties as well as meaningful information embedded in the document itself.

We can inspect the actual DOCX properties.

In [6]:
from docx import Document

docx_path = DATA_DIR / "q3_customer_support_operating_brief.docx"

docx = Document(docx_path)
properties = docx.core_properties

docx_metadata = {
    "title": properties.title,
    "subject": properties.subject,
    "author": properties.author,
    "keywords": properties.keywords,
    "comments": properties.comments,
    "created": properties.created.isoformat() if properties.created else None,
    "modified": properties.modified.isoformat() if properties.modified else None,
}

print(json.dumps(docx_metadata, indent=2))

{
  "title": "Q3 Customer Support Operating Brief",
  "subject": "Operations and service-level information",
  "author": "Customer Operations",
  "keywords": "support, SLA, refunds, operations",
  "comments": "generated by python-docx",
  "created": "2013-12-24T00:15:00+00:00",
  "modified": "2026-08-30T13:45:30+00:00"
}


We should also distinguish package metadata from content that appears in the document itself.

For this document, the visible content includes a document ID, version, preparation information, and document-control information.

Those are different provenance signals from the DOCX package properties.

In [7]:
for paragraph in docx.paragraphs:
    text = paragraph.text.strip()

    if text:
        print(f"[{paragraph.style.name}] {text}")

[Title] Q3 Customer Support Operating Brief
[Normal] Document ID: OPS-SUP-2026-Q3    Version: 1.0
[Normal] Prepared by Customer Operations | 22 August 2026
[Heading 1] Executive Summary
[Normal] The support team is prioritizing faster first responses, consistent handling of refund requests, and improved escalation for cases that require specialist review.
[Heading 1] Service Targets
[Normal] The following targets apply to standard customer-support requests during the quarter.
[Heading 1] Refund Handling
[Normal] Refund requests should include the order reference, purchase date, and reason for the request. Agents should verify eligibility before escalating a case.
[Heading 1] Escalation Notes
[Normal] Cases involving suspected payment fraud, account compromise, or conflicting customer records should be escalated to the appropriate specialist team.
[Heading 1] Document Control
[Normal] Owner: Customer Operations
Review frequency: Quarterly
Next scheduled review: 30 November 2026


The important lesson is:

> **Do not infer document semantics from file metadata when the document itself provides an explicit field.**

If the document contains a visible version field, that is usually more useful for document-level versioning than the filesystem modification timestamp.

## 8. HTML provenance

HTML provides another useful example because the source contains semantic elements.

Inspect the real document structure.

In [8]:
from bs4 import BeautifulSoup

html_path = DATA_DIR / "support_refunds.html"
html = html_path.read_text(encoding="utf-8")

soup = BeautifulSoup(html, "html.parser")

print("Title:", soup.title.get_text(strip=True) if soup.title else None)

article = soup.find("article")
print("Article:", article is not None)

if article:
    print("Article document ID:", article.get("data-document-id"))

Title: Support Knowledge Base — Refunds
Article: True
Article document ID: KB-REF-2026-07


The HTML article contains a source-level document identifier.

That identifier is more meaningful for provenance than the filename alone.

Semantic structure can therefore provide provenance information that would be lost if we immediately reduced the page to a single text string.

## 9. Markdown provenance

Markdown usually contains less formal file metadata than PDF, DOCX, or HTML.

That does not mean it has no useful provenance.

Inspect the actual file and keep the source path as ingestion metadata.

In [9]:
markdown_path = DATA_DIR / "internal_rag_engineering_guide.md"
markdown_text = markdown_path.read_text(encoding="utf-8")

print(markdown_text[:3000])

# Internal RAG Engineering Guide

**Document ID:** ENG-RAG-2026-014  
**Version:** 1.3  
**Owner:** AI Platform Engineering  
**Last Updated:** 20 August 2026

## Purpose

This guide describes the baseline architecture used for internal retrieval-augmented
generation experiments.

## Pipeline

The baseline pipeline contains the following stages:

1. Document ingestion
2. Chunking
3. Embedding generation
4. Vector indexing
5. Retrieval
6. Context construction
7. Generation
8. Evaluation

## Retrieval

The baseline retriever uses semantic similarity over document embeddings.

> Retrieval quality should be measured independently from answer quality.

## Metadata

Every indexed chunk should retain enough metadata to trace the result back to its
source document.

Recommended fields include:

```text
document_id
source
page
section
chunk_id
```

## Engineering Notes

Do not assume that a successful parser produced correct content. Ingestion output
should be inspected and validated before it 

For Markdown, the source filename/path may be the only reliable provenance available unless the document itself defines metadata.

That is why the ingestion pipeline needs to add its own metadata rather than expecting every source format to provide the same fields.

## 10. A common metadata envelope

A useful production design separates stable source information from extraction information.

For example:

```text
DocumentRecord
├── identity
│   ├── document_id
│   └── content_hash
│
├── source
│   ├── filename
│   ├── mime_type
│   └── source_uri
│
├── document_metadata
│   ├── title
│   ├── version
│   └── owner
│
└── ingestion
    ├── ingested_at
    ├── parser
    ├── parser_version
    └── extraction_method
```

Not every document will have every field.

Missing metadata should remain missing rather than being guessed.


## 11. Build a document-level provenance record

Now combine metadata that we can actually observe.

We will use the SHA-256 hash as the exact-artifact identifier.

In [10]:
def document_record(path: Path) -> dict:
    record = file_metadata(path)

    content_hash = sha256_file(path)

    record.update({
        "document_id": content_hash,
        "content_hash": content_hash,
        "ingestion": {
            "ingested_at": datetime.now(timezone.utc).isoformat(),
            "python_version": sys.version.split()[0],
            "platform": platform.platform(),
        },
    })

    return record

records = [document_record(path) for path in DOCUMENTS]

print(json.dumps(records[0], indent=2))

{
  "filename": "employee_travel_policy.pdf",
  "extension": ".pdf",
  "size_bytes": 44745,
  "mime_type": "application/pdf",
  "modified_at": "2026-08-30T15:15:53.263316+00:00",
  "document_id": "e269b160c4cb9ae314a3efadf7285d33829c6d661abb81ea09b95d107b10c9a9",
  "content_hash": "e269b160c4cb9ae314a3efadf7285d33829c6d661abb81ea09b95d107b10c9a9",
  "ingestion": {
    "ingested_at": "2026-09-03T15:27:14.401204+00:00",
    "python_version": "3.11.16",
    "platform": "Linux-6.8.0-79-generic-x86_64-with-glibc2.41"
  }
}


The `ingested_at`, Python version, and platform are **pipeline metadata**.

They are not claims about the source document.

Keeping those categories separate makes debugging much easier.

## 12. Add extraction provenance

A document can be extracted by different methods.

For example:

```text
native_pdf_pymupdf
pymupdf4llm
docx
html
ocr
```

The extraction method should travel with the extracted content.

In [11]:
def extraction_method(path: Path) -> str:
    suffix = path.suffix.lower()

    if suffix == ".pdf":
        return "pymupdf4llm"

    if suffix == ".docx":
        return "python-docx"

    if suffix == ".html":
        return "beautifulsoup"

    if suffix == ".md":
        return "plain-text"

    return "unknown"

for path in DOCUMENTS:
    print(path.name, "->", extraction_method(path))

employee_travel_policy.pdf -> pymupdf4llm
internal_rag_engineering_guide.md -> plain-text
q3_customer_support_operating_brief.docx -> python-docx
support_refunds.html -> beautifulsoup


This mapping is intentionally simple.

A real ingestion system should record the **actual code path used**, including fallback paths.

For example, if a PDF first goes through PyMuPDF4LLM and then gets escalated to OCR, the final provenance should not falsely say that native extraction alone produced the result.

## 13. Parser versions belong in provenance

Extraction output can change when parser versions change.

That means the parser version is part of the reproducibility story.

We can inspect the installed versions used by this notebook.

In [12]:
from importlib.metadata import version, PackageNotFoundError

def package_version(name: str):
    try:
        return version(name)
    except PackageNotFoundError:
        return None

VERSIONS = {
    "pymupdf": package_version("pymupdf"),
    "pymupdf4llm": package_version("pymupdf4llm"),
    "python-docx": package_version("python-docx"),
    "beautifulsoup4": package_version("beautifulsoup4"),
}

print(json.dumps(VERSIONS, indent=2))

{
  "pymupdf": "1.28.2",
  "pymupdf4llm": "1.28.2",
  "python-docx": "1.2.0",
  "beautifulsoup4": "4.15.0"
}


### Why this matters

Suppose a document was indexed six months ago.

Then you upgrade the PDF extraction dependency and re-ingest it.

If the resulting text changes, you need to know whether:

- the source changed,
- the parser changed,
- the extraction path changed,
- or the cleaning/chunking code changed.

Without provenance, that investigation becomes much harder.

## 14. Page-level provenance for PDFs

Document-level provenance is not enough for RAG.

If a user asks a question and the retriever returns a chunk, we need to know **where inside the document that chunk came from**.

PyMuPDF4LLM can provide page-level chunks.

In [13]:
import pymupdf4llm

pdf_chunks = pymupdf4llm.to_markdown(
    pdf_path,
    page_chunks=True
)

print("Page chunks:", len(pdf_chunks))

for i, chunk in enumerate(pdf_chunks):
    print(f"\n--- Chunk {i} ---")
    print("Keys:", chunk.keys())
    print(str(chunk)[:1200])

Page chunks: 1

--- Chunk 0 ---
Keys: dict_keys(['metadata', 'toc_items', 'page_boxes', 'text'])
defaultdict(<function make_page_chunk.<locals>.<lambda> at 0x71bad44280e0>, {'metadata': {'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'creator': '', 'producer': '', 'creationDate': '', 'modDate': '', 'trapped': '', 'encryption': None, 'file_path': 'data/employee_travel_policy.pdf', 'page_count': 1, 'page_number': 1}, 'toc_items': [], 'page_boxes': [{'index': 0, 'class': 'section-header', 'bbox': (98, 60, 497, 78), 'pos': (0, 45)}, {'index': 1, 'class': 'text', 'bbox': (60, 88, 391, 98), 'pos': (45, 116)}, {'index': 2, 'class': 'section-header', 'bbox': (60, 111, 124, 123), 'pos': (116, 132)}, {'index': 3, 'class': 'text', 'bbox': (60, 131, 532, 155), 'pos': (132, 320)}, {'index': 4, 'class': 'section-header', 'bbox': (60, 167, 171, 180), 'pos': (320, 344)}, {'index': 5, 'class': 'text', 'bbox': (60, 188, 530, 212), 'pos': (344, 487)}, {'index': 6, 'class':

The page boundary gives us a natural provenance anchor.

We can now associate extracted content with the source page before later chunking turns the page into smaller retrieval units.

## 15. Provenance should survive chunking

Imagine this source:

```text
Document
  └── Page 4
       └── Section: Refund Handling
            └── Chunk
```

After chunking, the chunk should still know:

```text
document_id
page_number
section
source
```

The text can change representation during the pipeline, but its source relationship should not disappear.

In [14]:
def make_page_provenance(
    document_id: str,
    filename: str,
    page_number: int,
    extraction_method: str,
    parser_versions: dict,
) -> dict:
    return {
        "document_id": document_id,
        "source_filename": filename,
        "page_number": page_number,
        "extraction_method": extraction_method,
        "parser_versions": parser_versions,
    }

pdf_record = document_record(pdf_path)

for page_number, page_chunk in enumerate(pdf_chunks, start=1):
    provenance = make_page_provenance(
        document_id=pdf_record["document_id"],
        filename=pdf_path.name,
        page_number=page_number,
        extraction_method="pymupdf4llm",
        parser_versions={
            "pymupdf": VERSIONS["pymupdf"],
            "pymupdf4llm": VERSIONS["pymupdf4llm"],
        },
    )

    print(provenance)
    if page_number == 2:
        break

{'document_id': 'e269b160c4cb9ae314a3efadf7285d33829c6d661abb81ea09b95d107b10c9a9', 'source_filename': 'employee_travel_policy.pdf', 'page_number': 1, 'extraction_method': 'pymupdf4llm', 'parser_versions': {'pymupdf': '1.28.2', 'pymupdf4llm': '1.28.2'}}


## 16. Source hierarchy

Provenance is easier to manage when we model the source hierarchy explicitly.

```text
Source artifact
      │
      ▼
Document
      │
      ├── page
      │    └── extracted region
      │
      └── page
           └── extracted region
                 │
                 ▼
               chunk
```

A chunk is therefore not an isolated piece of text.

It is a derived representation of a particular source region.

## 17. Document identity vs document version

These concepts should not be collapsed.

Consider a policy:

```text
Travel Policy
Version 1.0
```

and later:

```text
Travel Policy
Version 1.1
```

They may represent the same logical document but different versions.

A production system may therefore need:

```text
logical_document_id
version
content_hash
```

The exact strategy depends on the source system.

For example, a source repository may already provide a stable document ID. If it does, prefer that authoritative identifier instead of inventing a new logical ID from the filename.

## 18. Why timestamps are tricky

There are several different dates a pipeline might encounter:

```text
document authored
document approved
document effective
document reviewed
file created
file modified
file ingested
```

These dates have different meanings.

Do not rename `modified_at` to `updated_at` and assume they are equivalent.

For the Q3 support brief, for example, the document itself contains explicit review information. That is source content and should not be confused with the filesystem timestamp.

In [15]:
# Inspect the actual document-control section in the DOCX.

for paragraph in docx.paragraphs:
    if paragraph.style.name == "Heading 1":
        print("\n", paragraph.text)

    if paragraph.text.startswith("Owner:"):
        print(paragraph.text)


 Executive Summary

 Service Targets

 Refund Handling

 Escalation Notes

 Document Control
Owner: Customer Operations
Review frequency: Quarterly
Next scheduled review: 30 November 2026


This is why provenance design should preserve both:

- **source-declared dates**, when available, and
- **pipeline timestamps**, such as ingestion time.

They answer different questions.

## 19. Provenance for HTML sections

HTML gives us a useful example of section-level provenance.

The article has semantic elements that can be associated with extracted content.

In [16]:
if article:
    for element in article.find_all(["h1", "h2", "h3", "p"]):
        text = element.get_text(" ", strip=True)

        if text:
            print({
                "tag": element.name,
                "text": text[:180],
                "document_id": article.get("data-document-id")
            })

{'tag': 'p', 'text': 'Billing', 'document_id': 'KB-REF-2026-07'}
{'tag': 'h1', 'text': 'Refunds and Cancellations', 'document_id': 'KB-REF-2026-07'}
{'tag': 'p', 'text': 'Updated 18 August 2026', 'document_id': 'KB-REF-2026-07'}
{'tag': 'h2', 'text': 'Refund eligibility', 'document_id': 'KB-REF-2026-07'}
{'tag': 'p', 'text': 'Eligible customers can request a refund within 30 calendar days of the original purchase.', 'document_id': 'KB-REF-2026-07'}
{'tag': 'h2', 'text': 'Processing time', 'document_id': 'KB-REF-2026-07'}
{'tag': 'p', 'text': 'Approved refunds are normally processed within 7 business days .', 'document_id': 'KB-REF-2026-07'}
{'tag': 'h2', 'text': 'Contact support', 'document_id': 'KB-REF-2026-07'}
{'tag': 'p', 'text': 'Include your order reference when contacting the support team.', 'document_id': 'KB-REF-2026-07'}


The exact fields available depend on the source.

The general principle is:

> Preserve provenance at the finest useful granularity before flattening structure away.

## 20. What belongs in vector metadata?

A vector database payload should contain information useful for retrieval, filtering, citation, and debugging.

Good candidates often include:

```text
document_id
logical_document_id
version
source_filename
page_number
section
document_type
tenant_id
access-control identifiers
```

But do not automatically put the entire document record into every vector.

Large, duplicated, or volatile metadata increases storage and update costs.

## 21. Access control is part of provenance

In a multi-tenant RAG system, provenance is not only about citations.

The chunk must also remain associated with the correct security boundary.

For example:

```text
tenant_id
document_id
page_number
chunk_id
text
```

The retriever can then enforce:

```text
retrieve only chunks belonging to this tenant
```

Do not rely on the language model to decide whether a retrieved document is authorized.

## 22. Chunk identifiers

A chunk should have its own identifier in addition to the document identifier.

A simple conceptual structure is:

```text
document_id
    ↓
page_number
    ↓
chunk_index
    ↓
chunk_id
```

The exact ID scheme should be deterministic if you want reproducible indexing.

For example, a chunk ID can be derived from:

```text
document identity
+
content/version identity
+
location
+
chunking configuration
```

The important point is that changing the chunking configuration may legitimately produce different chunk identities.

In [17]:
def chunk_id(document_id: str, page_number: int, chunk_index: int) -> str:
    raw = f"{document_id}:{page_number}:{chunk_index}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()

example_document_id = pdf_record["document_id"]

for page_number in range(1, 3):
    for chunk_index in range(2):
        print(
            page_number,
            chunk_index,
            chunk_id(example_document_id, page_number, chunk_index)
        )

1 0 89ec86451074a6ab40719095a4fab0f686a67fd26000c8705e177be9671f5cb4
1 1 eab90c087fab77a092e0697ead2a36731df882f073a6a6c1e31eba48d6445184
2 0 e3bddec70f9ca046a27436325be3de995a56d91ae42246e4278a6460229a5334
2 1 58812e9e37578d6d85211becc9dea3a95f5dd78de8d4f8f3487759e5c6e6d71d


This is a deterministic example.

It is not a universal production ID scheme. A real system should decide whether chunk identity should change when the content changes, the parser changes, or the chunking configuration changes.

## 23. Provenance through the RAG pipeline

The metadata flow should look roughly like this:

```text
                SOURCE
                  │
                  ▼
              INGESTION
                  │
        ┌─────────┴─────────┐
        │                   │
     content            provenance
        │                   │
        └─────────┬─────────┘
                  ▼
               CLEANING
                  │
                  ▼
             STRUCTURE
                  │
                  ▼
              CHUNKING
                  │
        ┌─────────┴─────────┐
        │                   │
      vector             metadata
        │                   │
        └─────────┬─────────┘
                  ▼
              RETRIEVAL
                  │
                  ▼
              GENERATION
                  │
                  ▼
             CITATION / DEBUG
```

The key idea is that **content and provenance travel together**.

## 24. A compact provenance record

Here is an example of the shape we want later stages to consume.

This record uses values obtained from the real PDF and the actual ingestion environment.

In [18]:
page_number = 1

provenance_record = {
    "document_id": pdf_record["document_id"],
    "content_hash": pdf_record["content_hash"],
    "source_filename": pdf_path.name,
    "source_type": "pdf",
    "page_number": page_number,
    "extraction": {
        "method": "pymupdf4llm",
        "pymupdf_version": VERSIONS["pymupdf"],
        "pymupdf4llm_version": VERSIONS["pymupdf4llm"],
    },
    "ingested_at": pdf_record["ingestion"]["ingested_at"]
}

print(json.dumps(provenance_record, indent=2))

{
  "document_id": "e269b160c4cb9ae314a3efadf7285d33829c6d661abb81ea09b95d107b10c9a9",
  "content_hash": "e269b160c4cb9ae314a3efadf7285d33829c6d661abb81ea09b95d107b10c9a9",
  "source_filename": "employee_travel_policy.pdf",
  "source_type": "pdf",
  "page_number": 1,
  "extraction": {
    "method": "pymupdf4llm",
    "pymupdf_version": "1.28.2",
    "pymupdf4llm_version": "1.28.2"
  },
  "ingested_at": "2026-09-03T15:27:18.398464+00:00"
}


## 25. Validate provenance

Metadata is data too.

It should be validated just like extracted text.

At minimum, check that required provenance fields are present before indexing.

In [19]:
REQUIRED_PROVENANCE_FIELDS = {
    "document_id",
    "content_hash",
    "source_filename",
    "source_type",
}

def validate_provenance(record: dict) -> None:
    missing = REQUIRED_PROVENANCE_FIELDS - record.keys()

    if missing:
        raise ValueError(
            f"Missing provenance fields: {sorted(missing)}"
        )

validate_provenance(provenance_record)
print("Provenance validation passed.")

Provenance validation passed.


## 25. Provenance and citations

Later, when a retriever returns a chunk, provenance can support a response such as:

```text
Source: Employee Travel Policy
Page: 2
```

The important architectural point is that the citation should be generated from **stored provenance**, not guessed by the model after retrieval.

That makes citations auditable.

## 26. What not to do

### Do not use the filename as the entire provenance model

Filenames can change.

### Do not overwrite source metadata with pipeline metadata

`modified_at` and `ingested_at` are not the same thing.

### Do not invent missing document metadata

If the source does not tell us the author, keep it unknown.

### Do not discard page/section information before chunking

Once lost, reconstructing it reliably may be impossible.

### Do not let parser changes silently rewrite your index

Record parser versions and test extraction changes.

### Do not trust the LLM to reconstruct provenance

Store provenance as structured data.

## 27. Production design

A practical document record can evolve toward:

```text
Document
├── identity
│   ├── logical_document_id
│   ├── version
│   └── content_hash
│
├── source
│   ├── source_uri
│   ├── filename
│   └── mime_type
│
├── document_metadata
│   ├── title
│   ├── author
│   ├── owner
│   └── dates
│
├── ingestion
│   ├── ingested_at
│   ├── parser
│   ├── parser_version
│   ├── extraction_method
│   └── pipeline_version
│
└── derived content
    └── chunks
        ├── chunk_id
        ├── page
        ├── section
        └── text
```

This is not a rigid schema.

It is a mental model for keeping **identity, source metadata, pipeline metadata, and derived content separate**.

## 28. Key takeaways

1. **Metadata describes a document; provenance explains where derived content came from.**
2. **Source metadata and ingestion metadata must remain distinguishable.**
3. **A filename is not enough to identify a document.**
4. **Content hashes identify the exact artifact that was processed.**
5. **Logical document identity and document version are separate concepts.**
6. **Page and section provenance should survive chunking.**
7. **Parser and pipeline versions are important for reproducibility.**
8. **Access-control metadata must travel with retrieved content in multi-tenant systems.**
9. **Citations should come from structured provenance, not model guesses.**
10. **Missing metadata should remain missing rather than being fabricated.**

## What's Next?

## Tables & Structured Document

The next notebook goes deeper into tables and structured content across document formats. We will treat tables as data structures rather than simply as text that happens to contain columns.